# Identify player with the ball

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from matplotlib.lines import Line2D

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [2]:
DATA_DIR = Path("../data")

home_path = DATA_DIR / "Sample_Game_1_RawTrackingData_Home_Team.csv"
away_path = DATA_DIR / "Sample_Game_1_RawTrackingData_Away_Team.csv"
events_path = DATA_DIR / "Sample_Game_1_RawEventsData.csv"

In [3]:
home = pd.read_csv(home_path, skiprows=2)
away = pd.read_csv(away_path, skiprows=2)
events = pd.read_csv(events_path)

In [4]:
PITCH_LENGTH = 105
PITCH_WIDTH = 68

## Helper functions

In [5]:
def clean_tracking_columns(df):
    df = df.copy()
    cols = list(df.columns)

    for i in range(3, len(cols), 2):
        name = cols[i]
        cols[i] = f"{name}_x"
        cols[i + 1] = f"{name}_y"

    df.columns = cols
    return df

In [6]:
def convert_tracking_to_metres(df):
    df = df.copy()

    x_cols = [c for c in df.columns if c.endswith("_x")]
    y_cols = [c for c in df.columns if c.endswith("_y")]

    df[x_cols] = df[x_cols] * PITCH_LENGTH
    df[y_cols] = (1 - df[y_cols]) * PITCH_WIDTH

    return df

In [7]:
def convert_event_coordinates(df):
    df = df.copy()

    for col in ["Start X", "End X"]:
        df[col] = df[col] * PITCH_LENGTH

    for col in ["Start Y", "End Y"]:
        df[col] = (1 - df[col]) * PITCH_WIDTH

    return df

In [8]:
def active_players(frame, x_cols):
    players = []

    for x_col in x_cols:
        y_col = x_col.replace("_x", "_y")
        player = x_col.replace("_x", "")

        if pd.notna(frame[x_col]) and pd.notna(frame[y_col]):
            players.append((player, frame[x_col], frame[y_col]))

    return players

## Clean Data

In [9]:
home = clean_tracking_columns(home)
away = clean_tracking_columns(away)

In [10]:
home = convert_tracking_to_metres(home)
away = convert_tracking_to_metres(away)

In [11]:
events = convert_event_coordinates(events)

In [12]:
home_x_cols = [
    c for c in home.columns
    if c.startswith("Player") and c.endswith("_x")
]

away_x_cols = [
    c for c in away.columns
    if c.startswith("Player") and c.endswith("_x")
]

In [13]:
print(home.shape)
print(away.shape)
print(events.shape)

print(home["Ball_x"].min(), home["Ball_x"].max())
print(home["Ball_y"].min(), home["Ball_y"].max())

(145006, 33)
(145006, 33)
(1745, 14)
-4.7754 111.2958
-2.5717599999999976 70.71387999999999


## Who controls the ball?

Find the nearest player to the ball

In [14]:
def nearest_player_to_ball(home_frame, away_frame, home_x_cols, away_x_cols):

    ball = np.array([
        home_frame["Ball_x"],
        home_frame["Ball_y"]
    ])

    candidates = []

    for team, frame, x_cols in [
        ("Home", home_frame, home_x_cols),
        ("Away", away_frame, away_x_cols)
    ]:

        for x_col in x_cols:
            player = x_col.replace("_x", "")
            y_col = x_col.replace("_x", "_y")

            x = frame[x_col]
            y = frame[y_col]

            if pd.isna(x) or pd.isna(y):
                continue

            distance = np.linalg.norm(
                np.array([x, y]) - ball
            )

            candidates.append(
                (team, player, distance)
            )

    return min(candidates, key=lambda x: x[2])

Pick a time frame.

In [15]:
test_frame = 7519

home_frame = home.loc[home["Frame"] == test_frame].iloc[0]
away_frame = away.loc[away["Frame"] == test_frame].iloc[0]

Get the nearest player to the ball at the given time frame.

In [16]:
team, player, distance = nearest_player_to_ball(
    home_frame,
    away_frame,
    home_x_cols,
    away_x_cols
)

print(team, player, distance)

Away Player19 0.0


Apply possession threshold.

In [17]:
def identify_ball_carrier(
    home_frame,
    away_frame,
    home_x_cols,
    away_x_cols,
    max_distance=2.0
):
    team, player, distance = nearest_player_to_ball(
        home_frame,
        away_frame,
        home_x_cols,
        away_x_cols
    )

    if distance <= max_distance:
        return team, player, distance

    return None, None, distance

In [18]:
identify_ball_carrier(
    home_frame,
    away_frame,
    home_x_cols,
    away_x_cols
)

('Away', 'Player19', 0.0)

### Validate across many known time frames.

Use passes first.

In [19]:
pass_events = events[
    (events["Type"] == "PASS") &
    events["From"].notna() &
    events["Start Frame"].notna()
].copy()

In [20]:
results = []

for _, event in pass_events.iterrows():

    frame_id = int(event["Start Frame"])

    home_match = home[home["Frame"] == frame_id]
    away_match = away[away["Frame"] == frame_id]

    if home_match.empty or away_match.empty:
        continue

    home_frame = home_match.iloc[0]
    away_frame = away_match.iloc[0]

    predicted_team, predicted_player, distance = nearest_player_to_ball(
        home_frame,
        away_frame,
        home_x_cols,
        away_x_cols
    )

    results.append({
        "frame": frame_id,
        "true_team": event["Team"],
        "true_player": event["From"],
        "predicted_team": predicted_team,
        "predicted_player": predicted_player,
        "distance_to_ball": distance
    })

In [21]:
validation = pd.DataFrame(results)

validation["correct"] = (
    (validation["true_team"] == validation["predicted_team"]) &
    (validation["true_player"] == validation["predicted_player"])
)

validation["correct"].mean()

0.9774718397997497

This tells us how often the simple nearest-player rule correctly identifies the passer. 97.7% is good enough for us, but let's check the incorrect ones nonetheless.

In [22]:
incorrect = validation[~validation["correct"]].copy()

incorrect[
    [
        "frame",
        "true_team",
        "true_player",
        "predicted_team",
        "predicted_player",
        "distance_to_ball"
    ]
].head(20)

,frame,true_team,true_player,predicted_team,predicted_player,distance_to_ball
1,3,Away,Player21,Home,Player10,1.675053
31,6863,Away,Player21,Home,Player6,0.063308
95,20182,Away,Player15,Away,Player20,2.022304
108,21515,Away,Player19,Away,Player20,1.912900
112,22746,Away,Player21,Away,Player20,1.668873
128,23909,Away,Player19,Home,Player5,2.278105
160,29337,Away,Player18,Home,Player4,0.188425
168,30807,Away,Player19,Home,Player10,0.188227
188,35188,Away,Player20,Home,Player9,0.030753
306,59923,Home,Player12,Home,Player10,0.113530


Also compare distance distributions.

In [23]:
validation.groupby("correct")["distance_to_ball"].describe()

,count,mean,std,min,25%,50%,75%,max
correct,,,,,,,,
False,18.0,0.834651,0.865543,0.030753,0.114493,0.194599,1.673508,2.278105
True,781.0,0.118149,0.275612,0.000000,0.028808,0.080452,0.134707,5.369026


Calculate distance from the ball to the true passer.

In [24]:
def player_distance_to_ball(frame, player):
    return np.linalg.norm(
        np.array([
            frame[f"{player}_x"],
            frame[f"{player}_y"]
        ]) -
        np.array([
            frame["Ball_x"],
            frame["Ball_y"]
        ])
    )

In [25]:
results = []

for _, event in pass_events.iterrows():

    frame_id = int(event["Start Frame"])

    home_match = home[home["Frame"] == frame_id]
    away_match = away[away["Frame"] == frame_id]

    if home_match.empty or away_match.empty:
        continue

    home_frame = home_match.iloc[0]
    away_frame = away_match.iloc[0]

    predicted_team, predicted_player, distance = nearest_player_to_ball(
        home_frame, away_frame,
        home_x_cols, away_x_cols
    )

    true_frame = (
        home_frame if event["Team"] == "Home"
        else away_frame
    )

    true_distance = player_distance_to_ball(
        true_frame,
        event["From"]
    )

    results.append({
        "frame": frame_id,
        "true_team": event["Team"],
        "true_player": event["From"],
        "predicted_team": predicted_team,
        "predicted_player": predicted_player,
        "nearest_distance": distance,
        "true_player_distance": true_distance
    })

validation = pd.DataFrame(results)

validation["correct"] = (
    (validation["true_team"] == validation["predicted_team"]) &
    (validation["true_player"] == validation["predicted_player"])
)

Inspect only the failures.

In [26]:
validation[~validation["correct"]].sort_values(
    "true_player_distance"
)

,frame,true_team,true_player,predicted_team,predicted_player,nearest_distance,true_player_distance,correct
188,35188,Away,Player20,Home,Player9,0.030753,0.127906,False
601,113452,Away,Player20,Home,Player13,0.151080,0.156416,False
160,29337,Away,Player18,Home,Player4,0.188425,0.232596,False
498,92832,Home,Player3,Away,Player24,0.117382,0.280501,False
168,30807,Away,Player19,Home,Player10,0.188227,0.459191,False
31,6863,Away,Player21,Home,Player6,0.063308,1.139949,False
320,63994,Away,Player17,Home,Player8,0.084483,1.549138,False
1,3,Away,Player21,Home,Player10,1.675053,1.675053,False
95,20182,Away,Player15,Away,Player20,2.022304,2.593202,False
564,105645,Away,Player22,Home,Player5,1.995880,2.664375,False


This table is very informative. It shows two different failure types:

- Contested-ball situations: e.g. frame 35188 — true passer is 0.13 m from the ball, but an opponent is even closer at 0.03 m. Pure nearest-player logic can’t reliably distinguish control here.
- Event/tracking timing mismatches or ball-in-flight frames: some recorded passers are 3–25 m from the ball. We definitely should not force a ball carrier in those frames.

Identify a ball carrier only when the ball appears to be under control; otherwise label the frame as no carrier / ball in transit.

### Continuous Possession Detection

In [27]:
MAX_CONTROL_DISTANCE = 2.0
SWITCH_MARGIN = 0.5 #only change carrier when another 
                    #player is at least 0.5 m closer to the ball than the current carrier

In [28]:
def player_distances_to_ball(
    home_frame,
    away_frame,
    home_x_cols,
    away_x_cols
):
    ball = np.array([
        home_frame["Ball_x"],
        home_frame["Ball_y"]
    ])

    candidates = []

    for team, frame, x_cols in [
        ("Home", home_frame, home_x_cols),
        ("Away", away_frame, away_x_cols)
    ]:
        for x_col in x_cols:
            player = x_col.replace("_x", "")
            y_col = x_col.replace("_x", "_y")

            x = frame[x_col]
            y = frame[y_col]

            if pd.isna(x) or pd.isna(y):
                continue

            distance = np.linalg.norm(
                np.array([x, y]) - ball
            )

            candidates.append({
                "team": team,
                "player": player,
                "distance": distance
            })

    return candidates

In [29]:
def get_player_distance(candidates, team, player):
    for c in candidates:
        if c["team"] == team and c["player"] == player:
            return c["distance"]

    return np.inf

In [30]:
home_by_frame = home.set_index("Frame")
away_by_frame = away.set_index("Frame")

common_frames = sorted(
    home_by_frame.index.intersection(away_by_frame.index)
)

results = []

previous_team = None
previous_player = None

for frame_id in common_frames:

    home_frame = home_by_frame.loc[frame_id]
    away_frame = away_by_frame.loc[frame_id]

    candidates = player_distances_to_ball(
        home_frame,
        away_frame,
        home_x_cols,
        away_x_cols
    )

    nearest = min(candidates, key=lambda x: x["distance"])

    # Nobody appears to control the ball
    if nearest["distance"] > MAX_CONTROL_DISTANCE:
        carrier_team = None
        carrier_player = None
        carrier_distance = nearest["distance"]

    else:
        # No previous carrier -> use nearest player
        if previous_player is None:
            carrier_team = nearest["team"]
            carrier_player = nearest["player"]
            carrier_distance = nearest["distance"]

        else:
            previous_distance = get_player_distance(
                candidates,
                previous_team,
                previous_player
            )

            # Previous carrier is still close enough to control ball
            if previous_distance <= MAX_CONTROL_DISTANCE:

                # Keep them unless someone else is clearly closer
                if (
                    nearest["team"] != previous_team
                    or nearest["player"] != previous_player
                ) and (
                    nearest["distance"] + SWITCH_MARGIN
                    < previous_distance
                ):
                    carrier_team = nearest["team"]
                    carrier_player = nearest["player"]
                    carrier_distance = nearest["distance"]

                else:
                    carrier_team = previous_team
                    carrier_player = previous_player
                    carrier_distance = previous_distance

            # Previous player lost control -> assign nearest
            else:
                carrier_team = nearest["team"]
                carrier_player = nearest["player"]
                carrier_distance = nearest["distance"]

    results.append({
        "frame": frame_id,
        "time": home_frame["Time [s]"],
        "period": home_frame["Period"],
        "possession_team": carrier_team,
        "ball_carrier": carrier_player,
        "distance_to_ball": carrier_distance
    })

    previous_team = carrier_team
    previous_player = carrier_player

In [31]:
possession = pd.DataFrame(results)

possession.head()

,frame,time,period,possession_team,ball_carrier,distance_to_ball
0,1,0.04,1.0,Away,Player19,0.000000
1,2,0.08,1.0,NaN,NaN,3.685200
2,3,0.12,1.0,Home,Player10,1.675053
3,4,0.16,1.0,Home,Player10,0.744770
4,5,0.20,1.0,Home,Player10,1.892089


In [32]:
print(
    "Frames with no identified carrier:",
    possession["ball_carrier"].isna().mean()
)

Frames with no identified carrier: 0.20622594927106463


#### Validate against known time frames.

In [33]:
# Keep pass events with known passer and frame
pass_events = events[
    (events["Type"] == "PASS") &
    events["From"].notna() &
    events["Start Frame"].notna()
].copy()

pass_events["Start Frame"] = pass_events["Start Frame"].astype(int)

# Join detected possession to pass-start frames
pass_validation = pass_events[
    ["Start Frame", "Team", "From"]
].merge(
    possession,
    left_on="Start Frame",
    right_on="frame",
    how="left"
)

pass_validation["correct"] = (
    (pass_validation["Team"] == pass_validation["possession_team"]) &
    (pass_validation["From"] == pass_validation["ball_carrier"])
)

pass_validation["correct"].mean()

0.9737171464330413

In [34]:
pass_validation["ball_carrier"].isna().mean()

0.006257822277847309

In [35]:
pass_validation.loc[
    ~pass_validation["correct"],
    [
        "Start Frame",
        "Team",
        "From",
        "possession_team",
        "ball_carrier",
        "distance_to_ball"
    ]
].head(20)

,Start Frame,Team,From,possession_team,ball_carrier,distance_to_ball
1,3,Away,Player21,Home,Player10,1.675053
31,6863,Away,Player21,Home,Player6,0.063308
83,18252,Home,Player3,Away,Player17,0.763800
95,20182,Away,Player15,NaN,NaN,2.022304
100,20666,Away,Player17,NaN,NaN,3.483281
108,21515,Away,Player19,Away,Player20,1.912900
112,22746,Away,Player21,Away,Player20,1.668873
128,23909,Away,Player19,NaN,NaN,2.278105
132,24186,Away,Player16,NaN,NaN,2.475528
163,30442,Away,Player15,NaN,NaN,5.369026


#### Check no-carrier frames

In [36]:
no_carrier = possession[possession["ball_carrier"].isna()].copy()

no_carrier_frames = home[
    home["Frame"].isin(no_carrier["frame"])
].copy()

outside_pitch = (
    (no_carrier_frames["Ball_x"] < 0) |
    (no_carrier_frames["Ball_x"] > 105) |
    (no_carrier_frames["Ball_y"] < 0) |
    (no_carrier_frames["Ball_y"] > 68)
)

print("No-carrier frames:", len(no_carrier))
print("Ball outside pitch:", outside_pitch.mean())

No-carrier frames: 29904
Ball outside pitch: 0.017723381487426432


In [37]:
possession["no_carrier"] = possession["ball_carrier"].isna()

possession["segment"] = (
    possession["no_carrier"] != possession["no_carrier"].shift()
).cumsum()

no_carrier_runs = (
    possession[possession["no_carrier"]]
    .groupby("segment")
    .size()
)

# 25 fps → convert frames to seconds
(no_carrier_runs / 25).describe()

count    1339.000000
mean        0.893323
std         2.517827
min         0.040000
25%         0.320000
50%         0.640000
75%         1.120000
max        82.520000
dtype: float64

In [38]:
no_carrier_segments = (
    possession[possession["no_carrier"]]
    .groupby("segment")
    .agg(
        start_frame=("frame", "min"),
        end_frame=("frame", "max"),
        start_time=("time", "min"),
        end_time=("time", "max"),
        period=("period", "first"),
        frames=("frame", "size")
    )
)

no_carrier_segments["duration"] = (
    no_carrier_segments["frames"] / 25
)

no_carrier_segments.sort_values(
    "duration",
    ascending=False
).head(10)

,start_frame,end_frame,start_time,end_time,period,frames,duration
segment,,,,,,,
782,40258,42320,1610.32,1692.80,1.0,2063,82.52
778,38479,39359,1539.16,1574.36,1.0,881,35.24
914,50499,50654,2019.96,2026.16,1.0,156,6.24
1802,95014,95150,3800.56,3806.00,2.0,137,5.48
784,42363,42475,1694.52,1699.00,1.0,113,4.52
1810,95473,95583,3818.92,3823.32,2.0,111,4.44
184,8942,9048,357.68,361.92,1.0,107,4.28
2016,107590,107679,4303.60,4307.16,2.0,90,3.60
2670,143384,143472,5735.36,5738.88,2.0,89,3.56


Inspect the 82 second distruption of play.

In [39]:
events[
    (events["Start Time [s]"] >= 1610.32) &
    (events["Start Time [s]"] <= 1692.80)
][
    ["Team", "Type", "Subtype",
     "Start Time [s]", "End Time [s]",
     "From", "To"]
]

,Team,Type,Subtype,Start Time [s],End Time [s],From,To


No event happened.

In [40]:
long_gap = home[
    (home["Frame"] >= 40258) &
    (home["Frame"] <= 42320)
]

long_gap[
    ["Frame", "Time [s]", "Ball_x", "Ball_y"]
].head()

,Frame,Time [s],Ball_x,Ball_y
40257,40258,1610.32,67.04880,19.45140
40258,40259,1610.36,67.05090,19.45276
40259,40260,1610.40,67.05405,19.45412
40260,40261,1610.44,67.05720,19.45480
40261,40262,1610.48,67.06035,19.45616


The ball barely moves and is on the pitch.

In [41]:
possession[
    (possession["frame"] >= 40258) &
    (possession["frame"] <= 42320)
][
    ["frame", "time", "distance_to_ball"]
].describe()

,frame,time,distance_to_ball
count,2063.000000,2063.000000,2063.000000
mean,41289.000000,1651.560000,7.649637
std,595.681123,23.827245,2.063447
min,40258.000000,1610.320000,2.010072
25%,40773.500000,1630.940000,6.086983
50%,41289.000000,1651.560000,7.928405
75%,41804.500000,1672.180000,9.629420
max,42320.000000,1692.800000,10.947556


No one is closer to the ball than 2 meters. This segment is likely an inactive play.

### Function

In [42]:
def detect_possession(max_control_distance, switch_margin):

    results = []
    previous_team = None
    previous_player = None

    for frame_id in common_frames:

        home_frame = home_by_frame.loc[frame_id]
        away_frame = away_by_frame.loc[frame_id]

        candidates = player_distances_to_ball(
            home_frame,
            away_frame,
            home_x_cols,
            away_x_cols
        )

        nearest = min(candidates, key=lambda x: x["distance"])

        if nearest["distance"] > max_control_distance:
            carrier_team = None
            carrier_player = None
            carrier_distance = nearest["distance"]

        elif previous_player is None:
            carrier_team = nearest["team"]
            carrier_player = nearest["player"]
            carrier_distance = nearest["distance"]

        else:
            previous_distance = get_player_distance(
                candidates,
                previous_team,
                previous_player
            )

            if previous_distance <= max_control_distance:

                new_player = (
                    nearest["team"] != previous_team
                    or nearest["player"] != previous_player
                )

                clearly_closer = (
                    nearest["distance"] + switch_margin
                    < previous_distance
                )

                if new_player and clearly_closer:
                    carrier_team = nearest["team"]
                    carrier_player = nearest["player"]
                    carrier_distance = nearest["distance"]
                else:
                    carrier_team = previous_team
                    carrier_player = previous_player
                    carrier_distance = previous_distance

            else:
                carrier_team = nearest["team"]
                carrier_player = nearest["player"]
                carrier_distance = nearest["distance"]

        results.append({
            "frame": frame_id,
            "time": home_frame["Time [s]"],
            "period": home_frame["Period"],
            "possession_team": carrier_team,
            "ball_carrier": carrier_player,
            "distance_to_ball": carrier_distance
        })

        previous_team = carrier_team
        previous_player = carrier_player

    return pd.DataFrame(results)

#### Test parameters

In [43]:
CONTROL_DISTANCES = [1.0, 1.5, 2.0, 2.5]
SWITCH_MARGINS = [0.0, 0.1, 0.25, 0.5]

tuning_results = []

for control_distance in CONTROL_DISTANCES:
    for switch_margin in SWITCH_MARGINS:

        possession_test = detect_possession(
            control_distance,
            switch_margin
        )

        validation_test = pass_events[
            ["Start Frame", "Team", "From"]
        ].merge(
            possession_test,
            left_on="Start Frame",
            right_on="frame",
            how="left"
        )

        correct = (
            (validation_test["Team"] ==
             validation_test["possession_team"])
            &
            (validation_test["From"] ==
             validation_test["ball_carrier"])
        )

        tuning_results.append({
            "control_distance": control_distance,
            "switch_margin": switch_margin,
            "pass_accuracy": correct.mean(),
            "no_carrier_at_pass": validation_test[
                "ball_carrier"
            ].isna().mean(),
            "overall_no_carrier": possession_test[
                "ball_carrier"
            ].isna().mean()
        })

tuning_results = pd.DataFrame(tuning_results)

tuning_results.sort_values(
    "pass_accuracy",
    ascending=False
)

,control_distance,switch_margin,pass_accuracy,no_carrier_at_pass,overall_no_carrier
14,2.5,0.25,0.976220,0.002503,0.178758
15,2.5,0.50,0.976220,0.002503,0.178758
10,2.0,0.25,0.974969,0.006258,0.206226
12,2.5,0.00,0.974969,0.002503,0.178758
8,2.0,0.00,0.973717,0.006258,0.206226
11,2.0,0.50,0.973717,0.006258,0.206226
13,2.5,0.10,0.973717,0.002503,0.178758
2,1.0,0.25,0.972466,0.016270,0.263237
6,1.5,0.25,0.972466,0.013767,0.234521
9,2.0,0.10,0.972466,0.006258,0.206226


I’d rather be slightly conservative because we only want to calculate passing lanes when we’re reasonably confident someone actually possesses the ball.

Even though 2.5 / 0.25 has the highest pass-start accuracy (97.62%), it only improves over 2.0 / 0.25 (97.50%) by about one pass in the whole validation set.

The trade-off is more important: with 2.5 m, we assign a carrier in substantially more frames (17.9% no-carrier vs 20.6%). Some of those extra assignments could be balls in transit where nobody really has control.

In [44]:
MAX_CONTROL_DISTANCE = 2.0
SWITCH_MARGIN = 0.25

In [45]:
possession = detect_possession(
    MAX_CONTROL_DISTANCE,
    SWITCH_MARGIN
)

Save this final posession output.

In [46]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

possession.to_csv(
    PROCESSED_DIR / "possession.csv",
    index=False
)